In [0]:
%sql
SHOW CATALOGS;

catalog
chinook_azure_sql_catalog
chinook_catalog
sales_catalog
samples
system
workspace


In [0]:
%sql
CREATE CATALOG IF NOT EXISTS food_inspections;

In [0]:
%sql
SHOW CATALOGS;

catalog
chinook_azure_sql_catalog
chinook_catalog
food_inspections
sales_catalog
samples
system
workspace


In [0]:
%sql
USE CATALOG food_inspections;

CREATE SCHEMA IF NOT EXISTS raw;
CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS food_inspections.raw.chicago_files;
CREATE VOLUME IF NOT EXISTS food_inspections.raw.dallas_files;

In [0]:
%sql
SHOW SCHEMAS IN food_inspections;

databaseName
bronze
default
gold
information_schema
raw
silver


In [0]:
%sql
SHOW VOLUMES IN food_inspections.raw;

database,volume_name
raw,chicago_files
raw,dallas_files


In [0]:
print("Chicago files:")
for f in dbutils.fs.ls("/Volumes/food_inspections/raw/chicago_files/"):
    print(f"  {f.name}  ({f.size / 1024 / 1024:.1f} MB)")

print("\nDallas files:")
for f in dbutils.fs.ls("/Volumes/food_inspections/raw/dallas_files/"):
    print(f"  {f.name}  ({f.size / 1024 / 1024:.1f} MB)")

Chicago files:
  Food_Inspections_20260408.csv  (330.6 MB)

Dallas files:
  Restaurant_and_Food_Establishment_Inspections_(October_2016_to_January_2024)_20260408.csv  (192.8 MB)


In [0]:
df_chi_peek = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("multiLine", "true")
    .option("escape", '"')
    .csv("/Volumes/food_inspections/raw/chicago_files/")
    .limit(5)
)

print(f"Chicago columns ({len(df_chi_peek.columns)}):")
for i, c in enumerate(df_chi_peek.columns):
    print(f"  [{i}] {c}")

Chicago columns (17):
  [0] Inspection ID
  [1] DBA Name
  [2] AKA Name
  [3] License #
  [4] Facility Type
  [5] Risk
  [6] Address
  [7] City
  [8] State
  [9] Zip
  [10] Inspection Date
  [11] Inspection Type
  [12] Results
  [13] Violations
  [14] Latitude
  [15] Longitude
  [16] Location


In [0]:
df_dal_peek = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("multiLine", "true")
    .option("escape", '"')
    .csv("/Volumes/food_inspections/raw/dallas_files/")
    .limit(5)
)

print(f"Dallas columns ({len(df_dal_peek.columns)}):")
for i, c in enumerate(df_dal_peek.columns):
    print(f"  [{i}] {c}")

Dallas columns (114):
  [0] Restaurant Name
  [1] Inspection Type
  [2] Inspection Date
  [3] Inspection Score
  [4] Street Number
  [5] Street Name
  [6] Street Direction
  [7] Street Type
  [8] Street Unit
  [9] Street Address
  [10] Zip Code
  [11] Violation Description - 1
  [12] Violation Points - 1
  [13] Violation Detail - 1
  [14] Violation Memo - 1
  [15] Violation Description - 2
  [16] Violation Points - 2
  [17] Violation Detail - 2
  [18] Violation Memo - 2
  [19] Violation Description - 3
  [20] Violation Points - 3
  [21] Violation Detail - 3
  [22] Violation Memo - 3
  [23] Violation Description - 4
  [24] Violation Points - 4
  [25] Violation Detail - 4
  [26] Violation Memo - 4
  [27] Violation Description - 5
  [28] Violation Points - 5
  [29] Violation Detail - 5
  [30] Violation Memo - 5
  [31] Violation Description - 6
  [32] Violation Points - 6
  [33] Violation Detail - 6
  [34] Violation Memo - 6
  [35] Violation Description - 7
  [36] Violation Points - 7
  [3